Create Gold-level tables Based on Fraud Investigation

In [0]:
from datetime import datetime
import logging
import traceback
import os
import sys 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
%sql
USE financial_transactions_dataset.default

In [0]:
%sql
CREATE OR REPLACE TABLE gold_over_limit_or_income_fraud_transactions_per_year AS
WITH initial_fraud_data AS (SELECT
    client_id,
    YEAR(transaction_date) AS transaction_year,
    transaction_timestamp,
    credit_score,
    card_brand,
    card_type,
    card_on_dark_web,
    use_chip,
    mcc_code_description,
    fraud_flag,
    errors,
    amount,
    credit_limit,
    yearly_income,
    try_divide(amount,credit_limit)  AS percent_of_limit,
    try_divide(amount,yearly_income)  AS percent_of_income
FROM financial_transactions_dataset.default.silver_transactions_id
WHERE credit_limit != 0
AND amount > 0
),
over_limit_or_income_fraud_transactions_per_year_initial_data AS
(SELECT 
DISTINCT
transaction_year,
COUNT(transaction_year) AS total
FROM initial_fraud_data
WHERE percent_of_income > 0.99
OR percent_of_limit > 0.99
AND fraud_flag = 'YES'
GROUP BY transaction_year
),
over_limit_or_income_fraud_transactions_per_year AS (SELECT 
transaction_year,
SUM(total) AS total_fraud_transactions
FROM over_limit_or_income_fraud_transactions_per_year_initial_data
GROUP BY transaction_year
ORDER BY 1 ASC
)
SELECT * FROM over_limit_or_income_fraud_transactions_per_year;

CREATE OR REPLACE TABLE gold_over_limit_or_income_fraud_transactions_by_card_brand AS
WITH initial_fraud_data AS (SELECT
    client_id,
    YEAR(transaction_date) AS transaction_year,
    transaction_timestamp,
    credit_score,
    card_brand,
    card_type,
    card_on_dark_web,
    use_chip,
    mcc_code_description,
    fraud_flag,
    errors,
    amount,
    credit_limit,
    yearly_income,
    try_divide(amount,credit_limit)  AS percent_of_limit,
    try_divide(amount,yearly_income)  AS percent_of_income
FROM financial_transactions_dataset.default.silver_transactions_id
WHERE credit_limit != 0
AND amount > 0
),
over_limit_or_income_fraud_transactions_by_card_brand AS (SELECT 
card_brand,
card_type,
COUNT(*) AS total_transactions
FROM initial_fraud_data
WHERE percent_of_income > 0.99
OR percent_of_limit > 0.99
AND fraud_flag = 'YES'
GROUP BY card_brand, card_type
)
SELECT * FROM over_limit_or_income_fraud_transactions_by_card_brand;

CREATE OR REPLACE TABLE gold_fraud_by_action_per_year AS
WITH initial_fraud_data AS (SELECT
    client_id,
    YEAR(transaction_date) AS transaction_year,
    transaction_timestamp,
    credit_score,
    card_brand,
    card_type,
    card_on_dark_web,
    use_chip,
    mcc_code_description,
    fraud_flag,
    errors,
    amount,
    credit_limit,
    yearly_income,
    try_divide(amount,credit_limit)  AS percent_of_limit,
    try_divide(amount,yearly_income)  AS percent_of_income
FROM financial_transactions_dataset.default.silver_transactions_id
WHERE credit_limit != 0
AND amount > 0
),
fraud_by_action_per_year AS (SELECT 
transaction_year,
use_chip,
COUNT(*) AS total_transactions
FROM initial_fraud_data
WHERE percent_of_income > 0.99
OR percent_of_limit > 0.99
AND fraud_flag = 'YES'
GROUP BY transaction_year, use_chip
ORDER BY 1 ASC, 3 DESC
)
SELECT * FROM fraud_by_action_per_year;